In [1]:
# import os
# os.system("pkill -f jupyter")


In [2]:
import math
import re
from random import *
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os


In [3]:
#Set GPU device
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

os.environ['http_proxy']  = 'http://192.41.170.23:3128'
os.environ['https_proxy'] = 'http://192.41.170.23:3128'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device
print(device)

#make our work comparable if restarted the kernel
SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# torch.cuda.get_device_name(0)

cuda


# Task 1

# 1. Load Data (Task 1)

In [4]:
# !pip install datasets

In [5]:
from datasets import load_dataset

# load book corpus dataset
book_corpus = load_dataset("rojagtap/bookcorpus")

# Shuffle and take 100K subset
dataset = book_corpus["train"].shuffle(seed=42).select(range(100000))

print(dataset[0])


{'text': 'matt asked her about what movies and music she liked .'}


In [6]:
dataset

Dataset({
    features: ['text'],
    num_rows: 100000
})

In [7]:
sentences = dataset['text']
text = [x.lower() for x in sentences] #lower case
text = [re.sub("[.,!?\\-]", '', x) for x in text] #clean all symbols
# text

In [8]:
for sentence in text:
    print(sentence, "_____")
    words = sentence.split()
    print(words)
    break

matt asked her about what movies and music she liked  _____
['matt', 'asked', 'her', 'about', 'what', 'movies', 'and', 'music', 'she', 'liked']


## 1.1 Making Vocabs

In [9]:
from tqdm.auto import tqdm

# Combine everything into one to make vocab
word_list = list(set(" ".join(text).split()))
word2id = {'[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3}  # special tokens

# Create the word2id in a single pass
for i, w in tqdm(enumerate(word_list), desc="Creating word2id"):
    word2id[w] = i + 4  # because 0-3 are already occupied

# Precompute the id2word mapping (this can be done once after word2id is fully populated)
id2word = {v: k for k, v in word2id.items()}
vocab_size = len(word2id)
vocab_size

Creating word2id: 0it [00:00, ?it/s]

43992

In [10]:
vocab_size = len(word2id)

# List of all tokens for the whole text
token_list = []

# Process sentences more efficiently
for sentence in tqdm(text, desc="Processing sentences"):
    token_list.append([word2id[word] for word in sentence.split()])

# Now token_list contains the tokenized sentences

Processing sentences:   0%|          | 0/100000 [00:00<?, ?it/s]

In [11]:
#take a look at sentences
sentences[:2]

['matt asked her about what movies and music she liked .',
 "`` they are somewhere on this island , and we will find them . ''"]

In [12]:
#take a look at token_list
token_list[:2]

[[27646, 2333, 13708, 26843, 5543, 37967, 22730, 41012, 11699, 23122],
 [29205,
  31846,
  5007,
  9352,
  25261,
  11758,
  6730,
  22730,
  5443,
  23818,
  42665,
  18939,
  17085]]

In [13]:
#testing one sentence
for tokens in token_list[0]:
    print(id2word[tokens])

matt
asked
her
about
what
movies
and
music
she
liked


# 1.2 Data Loader

In [14]:
batch_size = 6
max_mask   = 5  # max masked tokens when 15% exceed, it will only be max_pred
max_len    = 1000 # maximum of length to be padded; 

In [15]:
def make_batch():
    batch = []
    positive = negative = 0

    while positive != batch_size/2 or negative != batch_size/2:
        a_idx, b_idx = randrange(len(sentences)), randrange(len(sentences))
        tokens_a, tokens_b = token_list[a_idx], token_list[b_idx]

        # 1) input ids: [CLS] A [SEP] B [SEP]
        input_ids = [word2id['[CLS]']] + tokens_a + [word2id['[SEP]']] + tokens_b + [word2id['[SEP]']]

        # 2) segment ids: 0 for A (+CLS/SEP), 1 for B (+SEP)
        segment_ids = [0]*(1+len(tokens_a)+1) + [1]*(len(tokens_b)+1)

        # 3) MLM masking (15%, at least 1, at most max_mask)
        n_pred = min(max_mask, max(1, int(round(len(input_ids)*0.15))))

        cand_pos = [i for i,tok in enumerate(input_ids)
                    if tok != word2id['[CLS]'] and tok != word2id['[SEP]']]
        shuffle(cand_pos)

        masked_tokens, masked_pos = [], []
        for pos in cand_pos[:n_pred]:
            masked_pos.append(pos)
            masked_tokens.append(input_ids[pos])

            r = random()
            if r < 0.8:                         # 80% -> [MASK]
                input_ids[pos] = word2id['[MASK]']
            elif r < 0.9:                       # 10% -> random token
                input_ids[pos] = randint(0, vocab_size-1)
            else:                                # 10% -> unchanged
                pass

        # 4) padding to max_len
        pad_len = max_len - len(input_ids)
        input_ids.extend([word2id['[PAD]']]*pad_len)
        segment_ids.extend([0]*pad_len)

        # pad masked info to max_mask
        if n_pred < max_mask:
            masked_tokens.extend([0]*(max_mask-n_pred))
            masked_pos.extend([0]*(max_mask-n_pred))

        # 5) NSP label (same notebook rule: positive if b is next sentence)
        if a_idx + 1 == b_idx and positive < batch_size/2:
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, True])
            positive += 1
        elif a_idx + 1 != b_idx and negative < batch_size/2:
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, False])
            negative += 1

    return batch


In [16]:
batch = make_batch()

In [17]:
#len of batch
len(batch)

6

In [18]:
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))
input_ids.shape, segment_ids.shape, masked_tokens.shape, masked_pos.shape, isNext.shape

(torch.Size([6, 1000]),
 torch.Size([6, 1000]),
 torch.Size([6, 5]),
 torch.Size([6, 5]),
 torch.Size([6]))

# 1.3. Model

## 1.3.1 Embedding

In [19]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, max_len, n_segments, d_model, device):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.seg_embed = nn.Embedding(n_segments, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.device = device

    def forward(self, x, seg):
        bsz, seq_len = x.size()
        pos = torch.arange(seq_len, device=self.device).unsqueeze(0).expand(bsz, seq_len)
        out = self.tok_embed(x) + self.pos_embed(pos) + self.seg_embed(seg)
        return self.norm(out)


## 3.2 Attention Mask

In [20]:
def get_attn_pad_mask(seq_q, seq_k, device):
    pad_mask = seq_k.eq(0).unsqueeze(1).to(device)      # (bs,1,len_k)
    return pad_mask.expand(seq_q.size(0), seq_q.size(1), seq_k.size(1))


Testing Attention Mask

In [21]:

print(get_attn_pad_mask(input_ids, input_ids, device).shape)

torch.Size([6, 1000, 1000])


## 1.3.2 Encoder

In [22]:
class EncoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, d_ff, d_k, device):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(n_heads, d_model, d_k, device)
        self.pos_ffn       = PoswiseFeedForwardNet(d_model, d_ff)

    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs, attn = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask) # enc_inputs to same Q,K,V
        enc_outputs = self.pos_ffn(enc_outputs) # enc_outputs: [batch_size x len_q x d_model]
        return enc_outputs, attn

In [23]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k, device):
        super(ScaledDotProductAttention, self).__init__()
        self.scale = torch.sqrt(torch.FloatTensor([d_k])).to(device)

    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / self.scale # scores : [batch_size x n_heads x len_q(=len_k) x len_k(=len_q)]
        scores.masked_fill_(attn_mask, -1e9) # Fills elements of self tensor with value where mask is one.
        attn = nn.Softmax(dim=-1)(scores)
        context = torch.matmul(attn, V)
        return context, attn

In [24]:
n_layers = 6    # number of Encoder of Encoder Layer
n_heads  = 8    # number of heads in Multi-Head Attention
d_model  = 768  # Embedding Size
d_ff = 768 * 4  # 4*d_model, FeedForward dimension
d_k = d_v = 64  # dimension of K(=Q), V
n_segments = 2

In [25]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_model, d_k, device):
        super(MultiHeadAttention, self).__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_k
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, self.d_v * n_heads)
        self.device = device
    def forward(self, Q, K, V, attn_mask):
        # q: [batch_size x len_q x d_model], k: [batch_size x len_k x d_model], v: [batch_size x len_k x d_model]
        residual, batch_size = Q, Q.size(0)
        # (B, S, D) -proj-> (B, S, D) -split-> (B, S, H, W) -trans-> (B, H, S, W)
        q_s = self.W_Q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)  # q_s: [batch_size x n_heads x len_q x d_k]
        k_s = self.W_K(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)  # k_s: [batch_size x n_heads x len_k x d_k]
        v_s = self.W_V(V).view(batch_size, -1, self.n_heads, self.d_v).transpose(1,2)  # v_s: [batch_size x n_heads x len_k x d_v]

        attn_mask = attn_mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1) # attn_mask : [batch_size x n_heads x len_q x len_k]

        # context: [batch_size x n_heads x len_q x d_v], attn: [batch_size x n_heads x len_q(=len_k) x len_k(=len_q)]
        context, attn = ScaledDotProductAttention(self.d_k, self.device)(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_v) # context: [batch_size x len_q x n_heads * d_v]
        output = nn.Linear(self.n_heads * self.d_v, self.d_model, device=self.device)(context)
        return nn.LayerNorm(self.d_model, device=self.device)(output + residual), attn # output: [batch_size x len_q x d_model]

In [26]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # (batch_size, len_seq, d_model) -> (batch_size, len_seq, d_ff) -> (batch_size, len_seq, d_model)
        return self.fc2(F.gelu(self.fc1(x)))

## 1.3.3 Putting them together

In [27]:
class BERT(nn.Module):
    def __init__(self, n_layers, n_heads, d_model, d_ff, d_k, n_segments, vocab_size, max_len, device):
        super().__init__()
        self.embedding = Embedding(vocab_size, max_len, n_segments, d_model, device)
        self.layers = nn.ModuleList([EncoderLayer(n_heads, d_model, d_ff, d_k, device)
                                     for _ in range(n_layers)])

        # NSP head
        self.fc = nn.Linear(d_model, d_model)
        self.activ = nn.Tanh()
        self.classifier = nn.Linear(d_model, 2)

        # MLM head
        self.linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

        # tie weights with token embedding
        self.decoder = nn.Linear(d_model, vocab_size, bias=False)
        self.decoder.weight = self.embedding.tok_embed.weight
        self.decoder_bias = nn.Parameter(torch.zeros(vocab_size))

        self.device = device
    
    
    def encode(self, input_ids, segment_ids=None):
        if segment_ids is None:
            segment_ids = torch.zeros_like(input_ids)
        
        out = self.embedding(input_ids, segment_ids)
        mask = get_attn_pad_mask(input_ids, input_ids, self.device)
        for layer in self.layers:
            out, _ = layer(out, mask)
            
        return out  # (batch, seq_len, d_model)
    def forward(self, input_ids, segment_ids, masked_pos):
        out = self.embedding(input_ids, segment_ids)
        mask = get_attn_pad_mask(input_ids, input_ids, self.device)

        for layer in self.layers:
            out, _ = layer(out, mask)

        # NSP: [CLS]
        pooled = self.activ(self.fc(out[:, 0]))
        logits_nsp = self.classifier(pooled)

        # MLM: gather masked positions
        masked_pos = masked_pos[:, :, None].expand(-1, -1, out.size(-1))
        h_masked = torch.gather(out, 1, masked_pos)
        h_masked = self.norm(F.gelu(self.linear(h_masked)))
        logits_lm = self.decoder(h_masked) + self.decoder_bias

        return logits_lm, logits_nsp


In [28]:
from tqdm.auto import tqdm

n_layers = 12    # number of Encoder of Encoder Layer
n_heads  = 12    # number of heads in Multi-Head Attention
d_model  = 768  # Embedding Size
d_ff = d_model * 4  # 4*d_model, FeedForward dimension
d_k = d_v = 64  # dimension of K(=Q), V
n_segments = 2

num_epoch = 1000
model = BERT(
    n_layers, 
    n_heads, 
    d_model, 
    d_ff, 
    d_k, 
    n_segments, 
    vocab_size, 
    max_len, 
    device
).to(device)  # Move model to GPU

In [29]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [30]:
batch = make_batch()
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))

# Move inputs to GPU
input_ids = input_ids.to(device)
segment_ids = segment_ids.to(device)
masked_tokens = masked_tokens.to(device)
masked_pos = masked_pos.to(device)
isNext = isNext.to(device)

# Wrap the epoch loop with tqdm
for epoch in tqdm(range(num_epoch), desc="Training Epochs"):
    optimizer.zero_grad()
    logits_lm, logits_nsp = model(input_ids, segment_ids, masked_pos)    
    #logits_lm: (bs, max_mask, vocab_size) ==> (6, 5, 34)
    #logits_nsp: (bs, yes/no) ==> (6, 2)

    #1. mlm loss
    #logits_lm.transpose: (bs, vocab_size, max_mask) vs. masked_tokens: (bs, max_mask)
    loss_lm = criterion(logits_lm.transpose(1, 2), masked_tokens) # for masked LM
    loss_lm = (loss_lm.float()).mean()
    #2. nsp loss
    #logits_nsp: (bs, 2) vs. isNext: (bs, )
    loss_nsp = criterion(logits_nsp, isNext) # for sentence classification
    
    #3. combine loss
    loss = loss_lm + loss_nsp
    if epoch % 100 == 0:
        print('Epoch:', '%02d' % (epoch), 'loss =', '{:.6f}'.format(loss))
    loss.backward()
    optimizer.step()

Training Epochs:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch: 00 loss = 104.844810
Epoch: 100 loss = 7.168372
Epoch: 200 loss = 3.802578
Epoch: 300 loss = 4.210914
Epoch: 400 loss = 3.472209
Epoch: 500 loss = 3.315502
Epoch: 600 loss = 3.398335
Epoch: 700 loss = 3.518300
Epoch: 800 loss = 3.876763
Epoch: 900 loss = 4.329379


In [31]:
# Save the model after training
torch.save(model.state_dict(), 'bert_model.pth')
print("Model saved to bert_model.pth")

Model saved to bert_model.pth


# 1.4 Inference

In [32]:
# Predict mask tokens ans isNext
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(batch[2]))
print([id2word[w.item()] for w in input_ids[0] if id2word[w.item()] != '[PAD]'])
input_ids = input_ids.to(device)
segment_ids = segment_ids.to(device)
masked_tokens = masked_tokens.to(device)
masked_pos = masked_pos.to(device)
isNext = isNext.to(device)

logits_lm, logits_nsp = model(input_ids, segment_ids, masked_pos)
#logits_lm:  (1, max_mask, vocab_size) ==> (1, 5, 34)
#logits_nsp: (1, yes/no) ==> (1, 2)

#predict masked tokens
#max the probability along the vocab dim (2), [1] is the indices of the max, and [0] is the first value
logits_lm = logits_lm.data.cpu().max(2)[1][0].data.numpy() 
#note that zero is padding we add to the masked_tokens
print('masked tokens (words) : ',[id2word[pos.item()] for pos in masked_tokens[0]])
print('masked tokens list : ',[pos.item() for pos in masked_tokens[0]])
print('masked tokens (words) : ',[id2word[pos.item()] for pos in logits_lm])
print('predict masked tokens list : ', [pos for pos in logits_lm])

#predict nsp
logits_nsp = logits_nsp.cpu().data.max(1)[1][0].data.numpy()
print(logits_nsp)
print('isNext : ', True if isNext else False)
print('predict isNext : ',True if logits_nsp else False)

['[CLS]', '``', 'do', 'you', '[MASK]', 'to', 'be', 'an', '[MASK]', "''", '[SEP]', 'setting', 'the', 'sample', 'on', '[MASK]', 'slide', 'she', 'peered', 'into', 'her', 'microscope', 'adjusting', 'the', 'focus', '[MASK]', '[SEP]']
masked tokens (words) :  ['have', 'knob', 'a', 'ass', '[PAD]']
masked tokens list :  [25968, 15463, 11179, 5463, 0]
masked tokens (words) :  ['[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
predict masked tokens list :  [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
0
isNext :  False
predict isNext :  False


# Task 2

# 2.2 Load Model

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) rebuild the SAME BERT architecture class you used in Task 1
# (same n_layers, n_heads, d_model, etc.)
model_bert = BERT(
    n_layers=n_layers,
    n_heads=n_heads,
    d_model=d_model,
    d_ff=d_ff,
    d_k=d_k,
    n_segments=2,
    vocab_size=vocab_size,
    max_len=max_len,
    device=device
).to(device)

# 2) load weights
state = torch.load("bert_model.pth", map_location=device)
model_bert.load_state_dict(state, strict=False)  # strict=False allows loading even if some keys are missing

for param in model_bert.embedding.parameters():
    param.requires_grad = False

for layer in model_bert.layers[:2]:   # freeze first N layers
    for p in layer.parameters():
        p.requires_grad = False

model_bert.eval()

BERT(
  (embedding): Embedding(
    (tok_embed): Embedding(43992, 768)
    (pos_embed): Embedding(1000, 768)
    (seg_embed): Embedding(2, 768)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (layers): ModuleList(
    (0-11): 12 x EncoderLayer(
      (enc_self_attn): MultiHeadAttention(
        (W_Q): Linear(in_features=768, out_features=768, bias=True)
        (W_K): Linear(in_features=768, out_features=768, bias=True)
        (W_V): Linear(in_features=768, out_features=768, bias=True)
      )
      (pos_ffn): PoswiseFeedForwardNet(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
      )
    )
  )
  (fc): Linear(in_features=768, out_features=768, bias=True)
  (activ): Tanh()
  (classifier): Linear(in_features=768, out_features=2, bias=True)
  (linear): Linear(in_features=768, out_features=768, bias=True)
  (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (de

# 2.1 Load SNLI 

In [34]:
snli = load_dataset("snli")

# Use train/validation
train_data = snli["train"]
val_data   = snli["validation"]


In [35]:
def is_valid(ex):
    return ex["label"] != -1 and ex["premise"] is not None and ex["hypothesis"] is not None

train_data = train_data.filter(is_valid)
val_data   = val_data.filter(is_valid)


In [36]:
def preprocess(s):
    s = s.lower()
    s = re.sub(r"[.,!?\\-]", "", s)
    return s

In [37]:
UNK_FALLBACK_ID = word2id.get("[UNK]", word2id["[MASK]"])


def encode_sentence(sent, max_len_single):
    words = preprocess(sent).split()
    ids = [word2id["[CLS]"]]
    for w in words:
        ids.append(word2id.get(w, UNK_FALLBACK_ID))
    ids.append(word2id["[SEP]"])

    # pad/truncate
    ids = ids[:max_len_single]
    pad_len = max_len_single - len(ids)
    ids += [word2id["[PAD]"]] * pad_len
    return ids


In [38]:
H = model_bert.encode(input_ids, segment_ids)

# 2.2 Model 

## 2.2.1 siamese network structures ecoder

In [39]:
def mean_pooling(token_embeddings, attention_mask):
    # token_embeddings: (bs, seq, dim)
    # attention_mask: (bs, seq) 1 for real tokens, 0 for PAD
    mask = attention_mask.unsqueeze(-1).float()  # (bs, seq, 1)
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


In [40]:
attn_mask = (input_ids != word2id["[PAD]"]).long()


In [41]:
class SoftmaxLossHead(nn.Module):
    def __init__(self, emb_dim, num_labels=3):
        super().__init__()
        self.classifier = nn.Linear(emb_dim * 3, num_labels)

    def forward(self, u, v):
        feats = torch.cat([u, v, torch.abs(u - v)], dim=1)
        return self.classifier(feats)


In [42]:
class SBERT(nn.Module):
    def __init__(self, bert_encoder, emb_dim, num_labels=3):
        super().__init__()
        self.bert = bert_encoder
        self.head = SoftmaxLossHead(emb_dim, num_labels)

    def sentence_embedding(self, sent_ids):
        # sent_ids: (bs, seq)
        seg = torch.zeros_like(sent_ids)  # single sentence -> segment 0
        H = self.bert.encode(sent_ids, seg)
        mask = (sent_ids != word2id["[PAD]"]).long()
        u = mean_pooling(H, mask)
        return u

    def forward(self, prem_ids, hypo_ids):
        u = self.sentence_embedding(prem_ids)
        v = self.sentence_embedding(hypo_ids)
        logits = self.head(u, v)
        return logits, u, v


## Prepare Dadtaloader

In [43]:
from torch.utils.data import Dataset, DataLoader
MAX_LEN_SINGLE = 128

class SNLIDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        ex = self.data[i]
        prem = encode_sentence(ex["premise"], MAX_LEN_SINGLE)
        hypo = encode_sentence(ex["hypothesis"], MAX_LEN_SINGLE)
        label = ex["label"]
        return torch.tensor(prem), torch.tensor(hypo), torch.tensor(label)

train_loader = DataLoader(SNLIDataset(train_data), batch_size=32, shuffle=True)
val_loader   = DataLoader(SNLIDataset(val_data), batch_size=32, shuffle=False)


In [44]:
from tqdm import tqdm

sbert = SBERT(model_bert, emb_dim=d_model, num_labels=3).to(device)

for epoch in tqdm(range(1, 4), desc="SBERT Training Epochs"):

    sbert.train()
    total_loss = 0

    for step, (prem_ids, hypo_ids, y) in tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch}",
        leave=False
    ):

        prem_ids = prem_ids.to(device)
        hypo_ids = hypo_ids.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits, u, v = sbert(prem_ids, hypo_ids)

        loss = criterion(logits, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(sbert.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

        # print every 200 batches (optional)
        if step % 200 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f}")

    print(f"Epoch {epoch} | train loss = {total_loss/len(train_loader):.4f}")


ch 1:   0%|          | 1/17168 [00:00<2:53:59,  1.64it/s]

epoch 1 step 0 loss 2.1761



ch 1:   1%|          | 201/17168 [01:38<2:19:56,  2.02it/s]

epoch 1 step 200 loss 2.4309



ch 1:   2%|▏         | 401/17168 [03:16<2:16:55,  2.04it/s]

...

epoch 3 step 16800 loss 1.9487



ch 3:  99%|█████████▉| 17001/17168 [2:13:14<01:15,  2.21it/s]

epoch 3 step 17000 loss 2.2600



SBERT Training Epochs: 100%|██████████| 3/3 [6:51:06<00:00, 8222.15s/it]  

Epoch 3 | train loss = 2.2167


In [47]:
print("Training finished.")

Training finished.


In [48]:
torch.save(sbert.state_dict(), "sbert_model.pth")

print("SBERT model saved!")

SBERT model saved!


In [52]:
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import torch

label_names = ["entailment", "neutral", "contradiction"]  # SNLI: 0,1,2

def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for prem_ids, hypo_ids, y in dataloader:
            prem_ids = prem_ids.to(device)
            hypo_ids = hypo_ids.to(device)
            y = y.to(device)

            logits, _, _ = model(prem_ids, hypo_ids)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    print("Accuracy:", accuracy_score(all_labels, all_preds))
    print(classification_report(all_labels, all_preds, target_names=label_names, digits=2))

# run it
evaluate(sbert, val_loader)

Accuracy: 0.3330623856939646
               precision    recall  f1-score   support

   entailment       0.00      0.00      0.00      3329
      neutral       0.00      0.00      0.00      3235
contradiction       0.33      1.00      0.50      3278

     accuracy                           0.33      9842
    macro avg       0.11      0.33      0.17      9842
 weighted avg       0.11      0.33      0.17      9842



/home/jupyter-st126425/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jupyter-st126425/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jupyter-st126425/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [55]:
# !pip install scikit-learn